# Material Management Notebook

This notebook manages liquid materials registered in the MADSci Resource Manager.

**Prerequisites:** `madsci_postgres` and `resource_manager` services must be running.

```bash
docker compose up -d --build madsci_postgres event_manager resource_manager
```

To use section 5 (apply calibration results from Data Manager), also start:

```bash
docker compose up -d --build madsci_mongodb data_manager
```

**Operations:**
1. Setup — connect to Resource Manager and Data Manager
2. List — show all registered materials
3. Add — register a new material
4. Update (Pattern A) — manually edit attributes
5. Update (Pattern B) — apply calibration results from Data Manager
6. Remove — delete a material

---
## 1. Setup

In [ ]:
from madsci.client.data_client import DataClient
from madsci.client.resource_client import ResourceClient
from madsci.common.types.resource_types import Consumable

# Resource Manager URL (default port 8003)
RESOURCE_MANAGER_URL = "http://localhost:8003/"

# Data Manager URL (default port 8004) — used in section 5
DATA_MANAGER_URL = "http://localhost:8004/"

# Resource class label used to identify high viscosity liquid materials
MATERIAL_CLASS = "high_viscosity_liquid"

# Current schema version — increment when attribute structure changes
CURRENT_SCHEMA_VERSION = "1.3"

resource_client = ResourceClient(resource_server_url=RESOURCE_MANAGER_URL)
data_client = DataClient(data_server_url=DATA_MANAGER_URL)

print(f"Resource Manager    : {RESOURCE_MANAGER_URL}")
print(f"Data Manager        : {DATA_MANAGER_URL}")
print(f"Material class      : {MATERIAL_CLASS}")
print(f"Schema version      : {CURRENT_SCHEMA_VERSION}")


2026-05-13T17:41:51.435376Z [info     ] EventClient initialized        client_name=madsci.client.resource_client event_server='Not configured' log_dir=.madsci\logs log_level=EventLogLevel.INFO madsci_version=0.7.0 platform=Windows-11-10.0.26200-SP0 python_version=3.13.2
2026-05-13T17:41:51.657139Z [info     ] EventClient initialized        client_name=madsci.client.data_client event_server='Not configured' log_dir=.madsci\logs log_level=EventLogLevel.INFO madsci_version=0.7.0 platform=Windows-11-10.0.26200-SP0 python_version=3.13.2


Resource Manager    : http://localhost:8003/
Data Manager        : http://localhost:8004/
Material class      : high_viscosity_liquid
Schema version      : 1.2


---
## Schema Version History

Track all structural changes to `attributes` here.  
When the structure changes, increment `CURRENT_SCHEMA_VERSION` and add a migration function in the next cell.

| Version | Date | Changes |
|---------|------|---------|
| 0.0 | 2026-05-05 | Initial schema |
| 1.0 | 2026-05-12 | dispensing_params: removed `g_per_rev`, renamed `max_stable_speed_rps` → `max_stable_speed_ml_per_min`, split `source_datapoint_id` → `source_type` + `source_datapoint_id`; added `suck_back_params` |
| 1.1 | 2026-05-12 | `dispensing_params`: added `device_name` as outer key to support multiple dispensers per material |
| 1.2 | 2026-05-13 | `physical_properties` → `physical_properties_nominal`（nominal のみ、measured 廃止）; `suck_back_params`（トップレベル）→ `dispensing_params[device][suck_back]`（デバイスレベル）; `dispensing_params[device][pressure]`: `max_stable_speed_ml_per_min` → `throughput` / `accuracy` サブセクション（各自 speed + density） |
| 1.3 | 2026-05-14 | `dispensing_params[device][pressure]`: added `min_shot` subsection（commanded_volume_ml, wait_s, measured_mass_mg） |


In [ ]:
# Schema definition: see .github/instructions/high_viscosity_liquid.instructions.md

# --- Migration functions ---
# Add new functions below when schema version is incremented.

def migrate_v0_0_to_v1_0(material):
    """Migration from v0.0 to v1.0."""
    ML_PER_REV = 0.05  # eco-PEN450 fixed displacement
    attrs = material.attributes or {}
    dispensing_params = attrs.get("dispensing_params", {})
    for key, params in dispensing_params.items():
        if not isinstance(params, dict):
            continue
        params.pop("g_per_rev", None)
        if "max_stable_speed_rps" in params:
            rps = params.pop("max_stable_speed_rps")
            params["max_stable_speed_ml_per_min"] = rps * 60.0 * ML_PER_REV if rps is not None else None
        if "source_datapoint_id" in params and "source_type" not in params:
            dp_id = params.pop("source_datapoint_id")
            params["source_type"] = "workflow" if dp_id else "manual"
            params["source_datapoint_id"] = dp_id
    attrs["dispensing_params"] = dispensing_params
    if "suck_back_params" not in attrs:
        attrs["suck_back_params"] = {"delay_s": None, "volume_ml": None}
    attrs["schema_version"] = "1.0"
    material.attributes = attrs
    return material


def migrate_v1_0_to_v1_1(material, default_device_name="high_viscosity_dispenser"):
    """Migration from v1.0 to v1.1.

    Wraps each pressure-keyed entry in dispensing_params under the device_name key.
    Existing entries are assumed to belong to default_device_name.
    """
    attrs = material.attributes or {}
    dispensing_params = attrs.get("dispensing_params", {})
    # Detect if already in v1.1 format (outer keys are device names, not pressure keys)
    already_migrated = any(
        isinstance(v, dict) and any(isinstance(vv, dict) for vv in v.values())
        for v in dispensing_params.values()
    )
    if not already_migrated and dispensing_params:
        attrs["dispensing_params"] = {default_device_name: dispensing_params}
    attrs["schema_version"] = "1.1"
    material.attributes = attrs
    return material


def migrate_v1_1_to_v1_2(material, default_device_name="high_viscosity_dispenser"):
    """Migration from v1.1 to v1.2.

    Changes:
    - physical_properties.nominal / .measured  →  physical_properties_nominal (nominal only)
    - suck_back_params (top-level)             →  dispensing_params[device][suck_back]
    - dispensing_params[device][pressure].max_stable_speed_ml_per_min
                                               →  throughput: {speed_ml_per_min, density_g_per_cm3}
                                                  accuracy:   {speed_ml_per_min, density_g_per_cm3}
      (accuracy speed = throughput speed if only one speed was recorded, density = same)
    """
    attrs = material.attributes or {}

    # physical_properties → physical_properties_nominal
    old_phys = attrs.pop("physical_properties", {})
    if old_phys:
        nominal = old_phys.get("nominal", {})
        attrs["physical_properties_nominal"] = nominal

    # suck_back_params → dispensing_params[device][suck_back]
    old_suck_back = attrs.pop("suck_back_params", None)
    dispensing_params = attrs.get("dispensing_params", {})
    if old_suck_back:
        device_params = dispensing_params.get(default_device_name, {})
        if "suck_back" not in device_params:
            device_params["suck_back"] = {
                "delay_s":   old_suck_back.get("delay_s"),
                "volume_ml": old_suck_back.get("volume_ml"),
            }
        dispensing_params[default_device_name] = device_params

    # max_stable_speed_ml_per_min → throughput / accuracy
    for device_name, device_params in dispensing_params.items():
        if not isinstance(device_params, dict):
            continue
        for key, pressure_params in device_params.items():
            if key == "suck_back" or not isinstance(pressure_params, dict):
                continue
            if "max_stable_speed_ml_per_min" in pressure_params:
                speed = pressure_params.pop("max_stable_speed_ml_per_min")
                pressure_params["throughput"] = {
                    "speed_ml_per_min":  speed,
                    "density_g_per_cm3": None,
                }
                pressure_params["accuracy"] = {
                    "speed_ml_per_min":  speed,
                    "density_g_per_cm3": None,
                }

    attrs["dispensing_params"] = dispensing_params
    attrs["schema_version"] = "1.2"
    material.attributes = attrs
    return material


def migrate_v1_2_to_v1_3(material):
    """Migration from v1.2 to v1.3.

    Changes:
    - dispensing_params[device][pressure]: added min_shot = None
    """
    attrs = material.attributes or {}
    dispensing_params = attrs.get("dispensing_params", {})
    for device_name, device_params in dispensing_params.items():
        if not isinstance(device_params, dict):
            continue
        for key, pressure_params in device_params.items():
            if key == "suck_back" or not isinstance(pressure_params, dict):
                continue
            if "min_shot" not in pressure_params:
                pressure_params["min_shot"] = None
    attrs["dispensing_params"] = dispensing_params
    attrs["schema_version"] = "1.3"
    material.attributes = attrs
    return material


print("Migration functions loaded.")


Migration functions loaded.


In [ ]:
# --- Migration: Apply to all registered materials ---
# Run this cell after loading the migration functions above.
# Each material is migrated step-by-step from its current version to CURRENT_SCHEMA_VERSION.

MIGRATION_STEPS = [
    ("0.0", "1.0", migrate_v0_0_to_v1_0),
    ("1.0", "1.1", migrate_v1_0_to_v1_1),
    ("1.1", "1.2", migrate_v1_1_to_v1_2),
    ("1.2", "1.3", migrate_v1_2_to_v1_3),
]

from requests.exceptions import HTTPError

try:
    all_materials = resource_client.query_resource(resource_class=MATERIAL_CLASS, multiple=True)
except HTTPError as e:
    if e.response is not None and e.response.status_code == 404:
        all_materials = []
    else:
        raise

if not all_materials:
    print("No materials found.")
else:
    for mat in all_materials:
        attrs = mat.attributes or {}
        current_version = attrs.get("schema_version", "0.0")
        applied = []
        for from_ver, to_ver, fn in MIGRATION_STEPS:
            if current_version == from_ver:
                mat = fn(mat)
                current_version = to_ver
                applied.append(f"{from_ver} → {to_ver}")
        if applied:
            resource_client.update_resource(mat)
            print(f"[{mat.resource_name}] migrated: {', '.join(applied)}")
        else:
            print(f"[{mat.resource_name}] already at v{current_version}, no migration needed.")


[test1] already at v1.2, no migration needed.
[test2] already at v1.2, no migration needed.
[__dispenser_check_test_material__] already at v1.2, no migration needed.


[Siltech F-60,000] already at v1.2, no migration needed.


---
## 2. List All Registered Materials

In [20]:
from requests.exceptions import HTTPError

# Query all resources with resource_class == MATERIAL_CLASS
try:
    materials = resource_client.query_resource(resource_class=MATERIAL_CLASS, multiple=True)
except HTTPError as e:
    if e.response is not None and e.response.status_code == 404:
        materials = []
    else:
        raise

if not materials:
    print("No materials registered yet.")
else:
    print(f"{len(materials)} material(s) registered:\n")
    for m in materials:
        attrs = m.attributes or {}
        print(f"  name       : {m.resource_name}")
        print(f"  id         : {m.resource_id}")
        print(f"  attributes : {attrs}")
        print()


2 material(s) registered:

  name       : test1
  id         : 01KQWSTP58SAFZXFHV149YAZGG


  attributes : {'schema_version': '1.2', 'product_info': {'material_name': 'test1', 'common_name': 'test1', 'manufacturer': 'SEKISUI', 'cas_number': '11111111', 'lot_number': 'ABC123', 'price_jpy_per_kg': '1500', 'application': 'hardener', 'chemical_class': 'amine'}, 'dispensing_params': {'high_viscosity_dispenser': {'suck_back': {'delay_s': None, 'volume_ml': None}}}, 'physical_properties_nominal': {'density_g_per_cm3': 1.54, 'viscosity_mPas': 30}}

  name       : Siltech F-60,000
  id         : 01KREK9KSQWTXZDCTQV01PW0DT
  attributes : {'schema_version': '1.2', 'product_info': {'material_name': 'Siltech F-60,000', 'common_name': 'Siltech F60K', 'manufacturer': 'SILTECK CORPORATION', 'cas_number': '63148-62-9', 'lot_number': '2002401', 'price_jpy_per_kg': 1000, 'application': 'Silicone'}, 'dispensing_params': {'high_viscosity_dispenser': {'0.1MPa': {'throughput': {'speed_ml_per_min': 6.0, 'density_g_per_cm3': 0.7333333333333334}, 'accuracy': {'speed_ml_per_min': 1.0, 'density_g_per_cm

---
## 3. Add a New Material

Edit `MATERIAL_NAME` and `ATTRIBUTES` before running.

In [ ]:
# --- Edit here ---
MATERIAL_NAME = "Siltech F-60,000"   # Unique name used as key in actions

ATTRIBUTES = {
    "schema_version": CURRENT_SCHEMA_VERSION,
    "product_info": {
        "material_name":    MATERIAL_NAME,          # Raw material name (required)
        "common_name":      "Siltech F60K",         # Common / trade name
        "manufacturer":     "SILTECK CORPORATION",  # Supplier name
        "cas_number":       "63148-62-9",           # CAS registry number
        "lot_number":       "2002401",              # Lot / batch number
        "price_jpy_per_kg": 1000,                   # Unit price [JPY/kg]
        "application":      "Silicone",             # e.g. "plasticizer", "pigment", "hardener"
        #"chemical_class":  "silicone",             # e.g. "amine", "silicone", "hydrocarbon"
    },
    "physical_properties_nominal": {               # Manufacturer's catalog / nominal values only
        "density_g_per_cm3": 0.98,                 # Density [g/cm³]
        "viscosity_mPas":   60000,                 # Viscosity [mPa·s]
    },
    "dispensing_params": {
        # Populated by calibration — see dispenser_check.ipynb section 3-3
        # "{device_name}": {                        e.g. "high_viscosity_dispenser"
        #     "suck_back": {                        device-level, pressure-independent
        #         "delay_s":   1.0,
        #         "volume_ml": 0.01,
        #     },
        #     "0.1MPa": {
        #         "throughput": {
        #             "speed_ml_per_min":  6.0,
        #             "density_g_per_cm3": 0.97,
        #         },
        #         "accuracy": {
        #             "speed_ml_per_min":  1.0,
        #             "density_g_per_cm3": 0.98,
        #         },
        #         "min_shot": {                     手動測定・手打ち（section 4 で更新）
        #             "commanded_volume_ml": 0.004, # 指示した吐出量 [mL]
        #             "wait_s":              2.0,   # 吐出後の待機時間 [s]
        #             "measured_mass_mg":    None,  # 実測された落下質量 [mg]
        #         },
        #         "calibrated_at":       "2026-05-13T10:00:00",
        #         "source_type":         "manual",
        #         "source_datapoint_id": None,
        #     }
        # }
        "high_viscosity_dispenser": {
            "suck_back": {
                "delay_s":   2,
                "volume_ml": 0.008,
            },
        },
    },
}
# -----------------

# Duplicate check — skip registration if the name is already in use
from requests.exceptions import HTTPError as _HTTPError
try:
    _existing = resource_client.query_resource(resource_name=MATERIAL_NAME, unique=True)
    print(f"ERROR: '{MATERIAL_NAME}' is already registered (id: {_existing.resource_id}).")
    print("Registration skipped. Use section 4 to update an existing material.")
except _HTTPError as _e:
    if _e.response is not None and _e.response.status_code == 404:
        # Not registered yet — proceed with registration
        new_material = Consumable(
            resource_name=MATERIAL_NAME,
            resource_class=MATERIAL_CLASS,
            attributes=ATTRIBUTES,
        )
        result = resource_client.add_resource(new_material)
        print(f"Registered: {result.resource_name} (id: {result.resource_id})")
        print(f"schema_version: {result.attributes.get('schema_version')}")
    else:
        raise


Registered: Siltech F-60,000 (id: 01KRH6X6P1KTER787F2ZGYWM0C)
schema_version: 1.2


---
## 4. Update (Pattern A) — Manual

Directly specify the new name or attributes to update.  
Use this when you want to edit basic material information.

In [ ]:

# --- Edit here ---
TARGET_NAME = "Siltech F-60,000"   # Current name of the material to update
NEW_NAME    = "Siltech F-60,000"   # New name (leave unchanged if not renaming)

# Only fill in the fields you want to change — others will be preserved.
PRODUCT_INFO_UPDATES = {
    # "material_name":    "Material_A",
    # "common_name":      "Mat-A",
    # "manufacturer":     "Sigma-Aldrich",
    # "cas_number":       "123-45-6",
    # "lot_number":       "ABC123",
    # "price_jpy_per_kg": 5000,
    # "application":      "plasticizer",
    # "chemical_class":   "silicone",
}
NOMINAL_PROPERTIES_UPDATES = {
    # Manufacturer's catalog / nominal values
    # "density_g_per_cm3": 1.05,
    # "viscosity_mPas":    50000,
}
SUCK_BACK_UPDATES = {
     #Device-level suck-back params. Key = device_name (from devices.settings.yaml)
     "high_viscosity_dispenser": {
         "delay_s":   3.0,
         "volume_ml": 0.008,
     },
}
MIN_SHOT_UPDATES = {
    # Key = device_name, value = {pressure_key: {commanded_volume_ml, wait_s, measured_mass_mg}}
    # "high_viscosity_dispenser": {
    #     "0.1MPa": {
    #         "commanded_volume_ml": 0.004,   # 指示した吐出量 [mL]
    #         "wait_s":              2.0,     # 吐出後の待機時間 [s]
    #         "measured_mass_mg":    0.35,    # 実測された落下質量 [mg]
    #     },
    # },
}
# -----------------

# Fetch the existing resource
target = resource_client.query_resource(
    resource_name=TARGET_NAME,
    resource_class=MATERIAL_CLASS,
    unique=True,
)

# Apply changes (merge into existing attributes)
attrs = target.attributes or {}
if PRODUCT_INFO_UPDATES:
    attrs["product_info"] = {**attrs.get("product_info", {}), **PRODUCT_INFO_UPDATES}
if NOMINAL_PROPERTIES_UPDATES:
    attrs["physical_properties_nominal"] = {
        **attrs.get("physical_properties_nominal", {}),
        **NOMINAL_PROPERTIES_UPDATES,
    }
if SUCK_BACK_UPDATES:
    dispensing_params = attrs.get("dispensing_params", {})
    for device_name, suck_back_vals in SUCK_BACK_UPDATES.items():
        device_params = dispensing_params.get(device_name, {})
        device_params["suck_back"] = {**device_params.get("suck_back", {}), **suck_back_vals}
        dispensing_params[device_name] = device_params
    attrs["dispensing_params"] = dispensing_params
if MIN_SHOT_UPDATES:
    dispensing_params = attrs.get("dispensing_params", {})
    for device_name, pressure_dict in MIN_SHOT_UPDATES.items():
        device_params = dispensing_params.get(device_name, {})
        for pressure_key, min_shot_vals in pressure_dict.items():
            pressure_params = device_params.get(pressure_key, {})
            pressure_params["min_shot"] = {**(pressure_params.get("min_shot") or {}), **min_shot_vals}
            device_params[pressure_key] = pressure_params
        dispensing_params[device_name] = device_params
    attrs["dispensing_params"] = dispensing_params

target.resource_name = NEW_NAME
target.attributes = attrs

updated = resource_client.update_resource(target)
print(f"Updated: {updated.resource_name} (id: {updated.resource_id})")
print(f"attributes: {updated.attributes}")


Updated: Siltech F-60,000 (id: 01KREK9KSQWTXZDCTQV01PW0DT)
attributes: {'schema_version': '1.2', 'product_info': {'material_name': 'Siltech F-60,000', 'common_name': 'Siltech F60K', 'manufacturer': 'SILTECK CORPORATION', 'cas_number': '63148-62-9', 'lot_number': '2002401', 'price_jpy_per_kg': 1000, 'application': 'Silicone'}, 'dispensing_params': {'high_viscosity_dispenser': {'0.1MPa': {'throughput': {'speed_ml_per_min': 6.0, 'density_g_per_cm3': 0.7333333333333334}, 'accuracy': {'speed_ml_per_min': 1.0, 'density_g_per_cm3': 0.9100000000000001}, 'calibrated_at': '2026-05-13T11:43:57', 'source_type': 'manual', 'source_datapoint_id': None}, 'suck_back': {'delay_s': 3.0, 'volume_ml': 0.008}}}, 'physical_properties_nominal': {'density_g_per_cm3': 0.98, 'viscosity_mPas': 60000}}


---
## 5. Update (Pattern B) — Apply Calibration Results

Fetch past calibration results for a material from the Data Manager,
select the result you want to apply, and write it back to the Resource Manager.

Calibration results are saved automatically by the Workcell Engine when the
`calibrate_dispenser` action is run via a Workflow (label: `json_result`, step_id: `calibrate_dispenser`).

**Prerequisites:** `madsci_mongodb` and `data_manager` must be running.

In [ ]:
# --- Edit here ---
TARGET_NAME = "Material_A"   # Material name to fetch calibration results for
# -----------------

# Fetch all calibration results for this material from Data Manager
# Results are saved automatically by the Workcell Engine (step_id = "calibrate_dispenser")
all_datapoints = data_client.query_datapoints({
    "ownership_info.step_id": "calibrate_dispenser",
    "value.material_name": TARGET_NAME,
})

if not all_datapoints:
    print(f"No calibration results found for '{TARGET_NAME}' in Data Manager.")
else:
    # Sort by timestamp (newest first) and display
    sorted_points = sorted(
        all_datapoints.values(),
        key=lambda d: d.data_timestamp,
        reverse=True,
    )
    print(f"Found {len(sorted_points)} calibration result(s) for '{TARGET_NAME}':\n")
    for i, dp in enumerate(sorted_points):
        optimal = dp.value.get("optimal", {})
        print(f"  [{i}] {dp.data_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"       device_name                 : {dp.value.get('device_name', 'N/A')}")
        print(f"       pressure_mpa                : {dp.value.get('pressure_mpa', 'N/A')}")
        print(f"       max_stable_speed_ml_per_min : {optimal.get('max_stable_speed_ml_per_min', 'N/A')}")
        print(f"       datapoint_id                : {dp.datapoint_id}")
        print()


In [ ]:

# --- Edit here ---
SELECTED_INDEX = 0         # Index of the calibration result to apply (from the list above)
PRESSURE_KEY   = "0.1MPa"  # Pressure condition key to store under dispensing_params
# -----------------

selected = sorted_points[SELECTED_INDEX]
optimal = selected.value.get("optimal", {})
max_stable_speed_ml_per_min = optimal["max_stable_speed_ml_per_min"]
device_name = selected.value.get("device_name")

if device_name is None:
    print("ERROR: device_name が結果に含まれていません。calibrate_dispenser を再実行してください。")
else:
    print(f"Applying calibration result [{SELECTED_INDEX}] under key ['{device_name}']['{PRESSURE_KEY}']:")
    print(f"  max_stable_speed_ml_per_min : {max_stable_speed_ml_per_min}")
    print(f"  datapoint_id                : {selected.datapoint_id}")

    # Fetch material from Resource Manager and update dispensing_params
    material = resource_client.query_resource(
        resource_name=TARGET_NAME,
        unique=True,
    )
    attrs = material.attributes or {}
    dispensing_params = attrs.get("dispensing_params", {})
    device_params = dispensing_params.get(device_name, {})
    device_params[PRESSURE_KEY] = {
        "max_stable_speed_ml_per_min": max_stable_speed_ml_per_min,
        "calibrated_at":               selected.data_timestamp.strftime("%Y-%m-%dT%H:%M:%S"),
        "source_type":                 "workflow",
        "source_datapoint_id":         selected.datapoint_id,
    }
    dispensing_params[device_name] = device_params
    attrs["dispensing_params"] = dispensing_params
    material.attributes = attrs

    updated = resource_client.update_resource(material)
    print(f"\nResource Manager updated: {updated.resource_name}")
    print(f"dispensing_params: {updated.attributes.get('dispensing_params')}")


---
## 6. Remove a Material

Set `TARGET_NAME` to the name of the material to remove.  
The resource is moved to history (soft delete) — it is not permanently deleted.

In [19]:
# --- Edit here ---
# Specify either TARGET_NAME or TARGET_ID — leave the other as None.
# If both are set, TARGET_ID takes priority.
TARGET_NAME = "__dispenser_check_test_material__"                           # Name of the material to remove (or None)
TARGET_ID   = None  # Resource ID of the material to remove (or None)
# -----------------

from requests.exceptions import HTTPError as _HTTPError

if TARGET_ID is not None:
    try:
        target = resource_client.get_resource(TARGET_ID)
    except _HTTPError as _e:
        if _e.response is not None and _e.response.status_code == 404:
            print(f"ERROR: No material found with id '{TARGET_ID}'.")
            target = None
        else:
            raise
elif TARGET_NAME is not None:
    try:
        target = resource_client.query_resource(
            resource_name=TARGET_NAME,
            resource_class=MATERIAL_CLASS,
            unique=True,
        )
    except _HTTPError as _e:
        if _e.response is not None and _e.response.status_code == 404:
            print(f"ERROR: No material found with name '{TARGET_NAME}'.")
            target = None
        else:
            raise
else:
    print("ERROR: Set either TARGET_NAME or TARGET_ID before running.")
    target = None

if target is not None:
    removed = resource_client.remove_resource(target.resource_id)
    print(f"Removed: {removed.resource_name} (id: {removed.resource_id})")


Removed: __dispenser_check_test_material__ (id: 01KQZNHMNDB49XC4368R6CPKPR)
